## **Processors: Deploying at Scale**

When we have an analysis, we would naturally like to scale it up to a far larger dataset in any practical scenario. More data means more processing time. We try to amend this fact by making use of parallel processing and distributed computing resources, which coffea can be deployed on rather naturally. Importantly, the details of our analysis are entirely independent of our deployment.

To expand our analysis, we will use coffea Processors. Processors are coffea’s way of encapsulating an analysis in a way that is deployment-neutral. Once you have a Coffea analysis, you can throw it into a processor and use any of a variety of executors to chunk it up and run it across distributed workers. This makes scale-out simple and dynamic for users.
  
Let’s start by writing a simple processor class that reads some CMS open data and plots a dimuon mass spectrum

In [ ]:
import hist
import correctionlib
import numpy as np
import awkward as ak
from coffea import processor
from coffea.analysis_tools import PackedSelection, Weights


def normalize(array: ak.Array):
    if array.ndim == 2:
        return ak.fill_none(ak.flatten(array), np.nan)
    else:
        return ak.fill_none(array, np.nan)


class MyProcessor(processor.ProcessorABC):
    def __init__(self, mode="virtual"):
        self._mode = mode

    def process(self, events):
        # dataset name
        dataset = events.metadata["dataset"]

        # save sum of gen weights
        sumw = ak.sum(events.genWeight) if hasattr(events, "genWeight") else len(events)

        # Object selection
        # Muons
        good_muons = (
            (events.Muon.pt >= 20)
            & (np.abs(events.Muon.eta) < 2.4)
            & events.Muon.tightId
            & (events.Muon.pfRelIso04_all <= 0.15)
        )
        muons = events.Muon[good_muons]

        # Dimuons
        pad_muons = ak.pad_none(muons, target=2)
        dimuons = pad_muons[:, 0] + pad_muons[:, 1]
        dimuons = dimuons.mask[(dimuons.mass > 60) & (dimuons.mass < 120)]

        # Event selection
        selections = PackedSelection()
        selections.add("trigger", events.HLT.IsoMu24)
        selections.add("at_least_one_vertex", events.PV.npvsGood > 0)
        selections.add("exactly_two_muons", ak.num(events.Muon) == 2)
        selections.add("opposite_charge_muons", ak.sum(events.Muon.charge, axis=1) == 0)
        selections.add("electron_veto", ak.num(events.Electron) == 0)

        region_selection = selections.all(*selections.names)
        pruned_events = events[region_selection]

        # create an instance of the Weights container
        weights_container = Weights(size=len(pruned_events), storeIndividual=True)
        if hasattr(events, "genWeight"):
            # generation weights
            weights_container.add("genweight", pruned_events.genWeight)
            # pileup weights
            pileup_weights = correctionlib.CorrectionSet.from_file("puWeights.json.gz")[
                "Collisions16_UltraLegacy_goldenJSON"
            ].evaluate(pruned_events.Pileup.nTrueInt, "nominal")
            weights_container.add("pileup", pileup_weights)

        # initialize output histogram
        dimuon_mass_histogram = hist.Hist(
            # dimuon mass axis
            hist.axis.Regular(
                bins=50,
                start=60,
                stop=120,
                name="dimuon_mass",
                label=r"$m_{\mu\mu}$ [GeV]",
            ),
            # set storage for weights
            hist.storage.Weight(),
        )

        # fill output histogram
        dimuon_mass_histogram.fill(
            dimuon_mass=normalize(dimuons.mass[region_selection]),
            weight=weights_container.weight(),
        )

        return {
            dataset: {
                "sumw": sumw,
                "mass": dimuon_mass_histogram,
            }
        }

    def postprocess(self, accumulator):
        pass

The `Processor` class gets deployed on an **executor**, which chunks up input data and feeds it in.

Currently, two local executors exist: 
* `iterative_executor`: The iterative executor simply processes each chunk of an input dataset in turn, using the current python thread.
* `futures_executor`: It employs python multiprocessing to spawn multiple python processes that process chunks in parallel on the machine
  
Thus, we have a pipeline: our input data is chunked, sent off to different workers which each execute the processor on their chunk, and then collected and converged once all workers finish processing.

![](https://i.imgur.com/nCxLquc.png)

In [ ]:
import warnings
from coffea.nanoevents import NanoAODSchema
warnings.filterwarnings("ignore", message="Missing cross-reference index ")


fileset = {
    "DYJetsToLL": {
        "files": {
            #"root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/DYJetsToLL_M-50_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/30000/0082C29D-E74C-024A-BE9B-97B29EE7A4A2.root": "Events",
            "DYJetsToLL_20UL16.root": "Events",
        },
    },
}

run = processor.Runner(
    #executor=processor.IterativeExecutor(compression=None),
    executor=processor.FuturesExecutor(workers=4, compression=None),
    schema=NanoAODSchema,
    savemetrics=True,
)

out, metrics = run(
    fileset,
    processor_instance=MyProcessor("virtual"),
)

In [ ]:
metrics

In [ ]:
out

In [ ]:
out['DYJetsToLL']['mass'].plot1d()

### **Normalize events to luminosity/cross section**

MC samples are generated with an arbitrary number of events that does not directly correspond to the amount of data collected by the experiment. In order to compare simulated events with data and to obtain physically meaningful event yields, the simulated samples must be normalized to the integrated luminosity of the data.

The expected number of events for a given process is given by
$$
N = L \times \sigma \times \epsilon,
$$
where $L$ is the integrated luminosity (which is related to the total number of proton–proton collisions recorded by the experiment), $\sigma$ is the cross section of the process, and $\epsilon$ is the selection efficiency. The selection efficiency is defined as the fraction of events that pass the analysis requirements,
$$
\epsilon = \frac{N_{\text{after}}}{N_{\text{before}}},
$$
where $N_{\text{before}}$ is the total number of generated events in the MC sample, given by the sum of generator weights (`sumw`), and $N_{\text{after}}$ is the weighted number of events remaining after applying all selection criteria.

This normalization is implemented by assigning a weight to each simulated event, such that the total weighted yield corresponds to the expected number of events for the given luminosity. Additional scale factors applied at later stages modify the number of events after selection, while the number of generated events, encoded in the sum of generator weights (`sumw`), remains unchanged.


In [ ]:
lumi = 16812.151722 # in units of 1/pb
xsec = 6025.6       # in units of pb

# compute luminosity/cross section weight
scale = (lumi * xsec) / out['DYJetsToLL']["sumw"]

# scale histogram to luminosity/cross section
dimuon_mass_histogram = out['DYJetsToLL']['mass'] * scale

# number of events
dimuon_mass_histogram.values(flow=True).sum()

In [ ]:
import mplhep as mh
import matplotlib.pyplot as plt

# set a CMS style
mh.style.use(mh.style.CMS)

# plot data histogram
mh.histplot(
    dimuon_mass_histogram,
    histtype="fill",
    stack=True,
    sort="yield",
    linewidth=0.7,
    edgecolor="k",
    label="DYJetsToLL"
)
mh.cms.text("OpenData", loc=1)
plt.ylabel("Events")
plt.legend();

**Exercise 1: (event-level cut)** Select transverse missing energy (MET) for events in which the condition $p_T(j) \geq 30$ GeV is met for at least two jets

hint: use the `ak.sum()` function

**Exercise 2:** Modify the previous processor to include 

* Muon selection:
    * Muons with $p_T>35$ GeV
    * Muons with $|\eta|<2.4$
    * Muons identified with a tight working point (`tightId`)
    * Muons with tight relative isolation (`pfRelIso03_all < 0.15`)
* Event selection:
    * Events triggered by `IsoMu24 OR IsoTkMu24`
* include additional muon scale factors:
muon ID (`NUM_TightID_DEN_TrackerMuons`)

![](https://i.imgur.com/crYp3Ef.png)

Iso (`NUM_TightRelIso_DEN_TightIDandIPCut`) 

![](https://i.imgur.com/Oz1dSkI.png)

and TriggerIso (`NUM_IsoMu24_or_IsoTkMu24_DEN_CutBasedIdTight_and_PFIsoTight`) scale factors

![](https://i.imgur.com/w8ElQY8.png)

Read the SFs from `muon_Z.json.gz'`.

* Add additional histograms to include leading and subleading muon $p_T$, $\eta$ and $\phi$ (normalized to lumi/cross section)




**Hint 1** Corrections only apply to 1D arrays. Therefore, you must flatten (use `ak.flatten()` and `ak.num()`) the arrays to calculate the SFs and then unflatten them to the original shape (use `ak.unflatten()`)
    
**Hint 2** Note that in some corrections the variables are limited to a defined range (use `np.clip()` so values outside the interval are clipped to the interval edges). 
    
**Hint 3:** The event scale factor is obtained by multiplying the scale factors of each object in the event (use `ak.prod()`).

In [ ]:
cset = correctionlib.CorrectionSet.from_file("muon_Z.json.gz")

In [ ]:
cset["NUM_TightID_DEN_TrackerMuons"].evaluate(2.1, 50., "nominal")

In [ ]:
cset["NUM_TightRelIso_DEN_TightIDandIPCut"].evaluate(2.1, 50., "nominal")

In [ ]:
cset["NUM_IsoMu24_or_IsoTkMu24_DEN_CutBasedIdTight_and_PFIsoTight"].evaluate(2.1, 50., "nominal")